# PI Few-Shot Raw CSI Supervised Contrastive Training

Training-only notebook. It trains the raw CSI encoder with domain-aware supervised contrastive loss, saves crash-recovery checkpoints during training, and saves the best model as the canonical `model.pt` used by the evaluation pipeline.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RAW_CSI_DIR = PROJECT_ROOT / "data" / "raw_csi_traces_pi"

## Run And Training Configuration

In [2]:
import gc
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display
from torch.cuda import is_available
from tqdm.auto import tqdm

from wifi_doppler.data.raw_csi_dataset import RawCsiWindowDataset
from wifi_doppler.evaluation.fewshot import window_true_labels_from_recordings
from wifi_doppler.experiments.artifacts import plot_step_curves, save_checkpoint, save_json
from wifi_doppler.experiments.runs import ensure_run, refresh_run_checkpoint, run_checkpoint_path, run_dir, training_dir
from wifi_doppler.models.raw_csi import RawCsiTemporalEncoder, count_trainable_parameters
from wifi_doppler.training.contrastive import contrastive_step, load_contrastive_batch, sample_contrastive_indices
from wifi_doppler.training.prototypical import evaluate_cross_dataset_episodes, evaluate_same_dataset_episodes, window_domains_from_recordings

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if is_available() else "cpu"

MODEL_RUN_ID = "20260531_raw_csi_supcon_weighted_domain"
MODEL_LABEL = "Raw CSI SupCon, weighted domain positives"
MODEL_KEY = "raw_csi_proto"
REPRESENTATION = "raw_csi"
BUILDER = "raw_csi_proto"
TRAINING_OBJECTIVE = "supervised_contrastive"
EPISODE_STYLE = "weighted-domain-positives"

PERSONS = ("p03", "p05", "p06", "p07", "p08", "p09", "p10", "p11", "p12", "p13")
TRAIN_SCENARIOS = ("PI-1a", "PI-2a", "PI-3a")
TARGET_SCENARIOS = ("PI-4a",)
WINDOW_SIZE = 340
WINDOW_STRIDE = 30
SPLIT_GUARD = 31

SOURCE_TRAIN_SPLIT = (0.0, 0.6)
SOURCE_VAL_SPLIT = (0.8, 1.0)
TARGET_ENROLLMENT_SPLIT = (0.0, 0.6)
TARGET_QUERY_SPLIT = (0.8, 1.0)

TOTAL_STEPS = 3000
EVAL_EVERY_STEPS = 100
SAVE_EVERY_STEPS = 25
LIVE_PLOT_EVERY_STEPS = 1
VAL_EPISODES_PER_EVAL = 20

CLASSES_PER_BATCH = len(PERSONS)
DOMAINS_PER_CLASS = 3
SAMPLES_PER_CLASS_PER_DOMAIN = 2
SUPCON_LR = 1e-4
SUPCON_TEMPERATURE = 0.1

SAME_DOMAIN_POSITIVE_WEIGHT = 0.25
CROSS_DOMAIN_POSITIVE_WEIGHT = 1.0

VAL_N_WAY = len(PERSONS)
VAL_K_SHOT = 5
VAL_Q_QUERY = 8
VAL_METRIC = "cosine"
VAL_TEMPERATURE = 0.1
EARLY_STOP_METRIC = "target_val_acc"
EARLY_STOP_PATIENCE_EVALS = 6
EARLY_STOP_MIN_DELTA = 0.0

EMBEDDING_DIM = 128
RAW_IN_CHANNELS = 4 * 242
RAW_CHANNEL_MIXER_DIM = 128
RAW_HIDDEN_DIM = 256

run_path = ensure_run(
    PROJECT_ROOT,
    model_run_id=MODEL_RUN_ID,
    label=MODEL_LABEL,
    model_key=MODEL_KEY,
    representation=REPRESENTATION,
    builder=BUILDER,
    training_objective=TRAINING_OBJECTIVE,
    episode_style=EPISODE_STYLE,
    notes="Domain-aware weighted supervised contrastive training. Same-person cross-domain positives have full weight; same-person same-domain positives have reduced weight.",
)
train_artifact_dir = training_dir(PROJECT_ROOT, MODEL_RUN_ID)
train_artifact_dir.mkdir(parents=True, exist_ok=True)

print("Run directory:", run_path)
print("Raw CSI directory:", RAW_CSI_DIR)
print("Train scenarios:", TRAIN_SCENARIOS)
print("Target scenarios:", TARGET_SCENARIOS)
print("Labels:", PERSONS)
print("Device:", device)

Run directory: C:\Users\gianm\Development\wifi-doppler-har\experiments\runs\20260531_raw_csi_supcon_weighted_domain
Raw CSI directory: C:\Users\gianm\Development\wifi-doppler-har\data\raw_csi_traces_pi
Train scenarios: ('PI-1a', 'PI-2a', 'PI-3a')
Target scenarios: ('PI-4a',)
Labels: ('p03', 'p05', 'p06', 'p07', 'p08', 'p09', 'p10', 'p11', 'p12', 'p13')
Device: cuda


## Build Datasets

In [ ]:
raw_train_dataset = RawCsiWindowDataset(
    RAW_CSI_DIR,
    scenarios=TRAIN_SCENARIOS,
    split=SOURCE_TRAIN_SPLIT,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    split_guard=SPLIT_GUARD,
    labels=PERSONS,
    flatten_channels=True,
    cache_traces=True,
)

raw_source_val_dataset = RawCsiWindowDataset(
    RAW_CSI_DIR,
    scenarios=TRAIN_SCENARIOS,
    split=SOURCE_VAL_SPLIT,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    split_guard=SPLIT_GUARD,
    labels=PERSONS,
    flatten_channels=True,
    cache_traces=True,
)

raw_target_enrollment_dataset = RawCsiWindowDataset(
    RAW_CSI_DIR,
    scenarios=TARGET_SCENARIOS,
    split=TARGET_ENROLLMENT_SPLIT,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    split_guard=SPLIT_GUARD,
    labels=PERSONS,
    flatten_channels=True,
    cache_traces=True,
)

raw_target_query_dataset = RawCsiWindowDataset(
    RAW_CSI_DIR,
    scenarios=TARGET_SCENARIOS,
    split=TARGET_QUERY_SPLIT,
    window_size=WINDOW_SIZE,
    window_stride=WINDOW_STRIDE,
    split_guard=SPLIT_GUARD,
    labels=PERSONS,
    flatten_channels=True,
    cache_traces=True,
)

raw_train_labels = window_true_labels_from_recordings(raw_train_dataset)
raw_train_domains = window_domains_from_recordings(raw_train_dataset)
raw_source_val_labels = window_true_labels_from_recordings(raw_source_val_dataset)
raw_target_enrollment_labels = window_true_labels_from_recordings(raw_target_enrollment_dataset)
raw_target_query_labels = window_true_labels_from_recordings(raw_target_query_dataset)

print("Raw source train windows:", len(raw_train_dataset))
print("Raw source train domains:", sorted(set(raw_train_domains.tolist())))
print("Raw source validation windows:", len(raw_source_val_dataset))
print("Raw target enrollment windows:", len(raw_target_enrollment_dataset))
print("Raw target query windows:", len(raw_target_query_dataset))
print("Raw sample shape:", tuple(raw_train_dataset[0][0].shape))
print("Contrastive batch size:", CLASSES_PER_BATCH * DOMAINS_PER_CLASS * SAMPLES_PER_CLASS_PER_DOMAIN)

## Model And Validation Helpers

In [ ]:
def build_raw_encoder(device: str | torch.device):
    model = RawCsiTemporalEncoder(
        in_channels=RAW_IN_CHANNELS,
        embedding_dim=EMBEDDING_DIM,
        channel_mixer_dim=RAW_CHANNEL_MIXER_DIM,
        hidden_dim=RAW_HIDDEN_DIM,
        normalize=True,
    ).to(device)
    with torch.no_grad():
        dummy = raw_target_enrollment_dataset[0][0].unsqueeze(0).to(device)
        _ = model.forward_embedding(dummy)
    print("Raw CSI SupCon trainable parameters:", count_trainable_parameters(model))
    return model


def eval_source_episodes(model, rng):
    return evaluate_same_dataset_episodes(
        model,
        raw_source_val_dataset,
        raw_source_val_labels,
        device=device,
        n_episodes=VAL_EPISODES_PER_EVAL,
        n_way=VAL_N_WAY,
        k_shot=VAL_K_SHOT,
        q_query=VAL_Q_QUERY,
        rng=rng,
        metric=VAL_METRIC,
        temperature=VAL_TEMPERATURE,
        fast_by_recording=True,
    )


def eval_target_episodes(model, rng):
    return evaluate_cross_dataset_episodes(
        model,
        raw_target_enrollment_dataset,
        raw_target_enrollment_labels,
        raw_target_query_dataset,
        raw_target_query_labels,
        device=device,
        n_episodes=VAL_EPISODES_PER_EVAL,
        n_way=VAL_N_WAY,
        k_shot=VAL_K_SHOT,
        q_query=VAL_Q_QUERY,
        rng=rng,
        metric=VAL_METRIC,
        temperature=VAL_TEMPERATURE,
        fast_by_recording=True,
    )


def show_live_curves(history):
    fig, _ = plot_step_curves(
        history,
        loss_keys=["contrastive_train_loss", "source_val_loss", "target_val_loss"],
        acc_keys=["contrastive_train_acc", "source_val_acc", "target_val_acc"],
        title="Raw CSI supervised contrastive training",
    )
    clear_output(wait=True)
    display(fig)
    plt.close(fig)

## Train

In [ ]:
supcon_config = {
    "raw_csi_dir": RAW_CSI_DIR,
    "persons": PERSONS,
    "train_scenarios": TRAIN_SCENARIOS,
    "target_scenarios": TARGET_SCENARIOS,
    "source_train_split": SOURCE_TRAIN_SPLIT,
    "source_val_split": SOURCE_VAL_SPLIT,
    "target_enrollment_split": TARGET_ENROLLMENT_SPLIT,
    "target_query_split": TARGET_QUERY_SPLIT,
    "window_size": WINDOW_SIZE,
    "window_stride": WINDOW_STRIDE,
    "split_guard": SPLIT_GUARD,
    "raw_in_channels": RAW_IN_CHANNELS,
    "raw_channel_mixer_dim": RAW_CHANNEL_MIXER_DIM,
    "raw_hidden_dim": RAW_HIDDEN_DIM,
    "proto_embedding_dim": EMBEDDING_DIM,
    "total_steps": TOTAL_STEPS,
    "eval_every_steps": EVAL_EVERY_STEPS,
    "save_every_steps": SAVE_EVERY_STEPS,
    "val_episodes_per_eval": VAL_EPISODES_PER_EVAL,
    "classes_per_batch": CLASSES_PER_BATCH,
    "domains_per_class": DOMAINS_PER_CLASS,
    "samples_per_class_per_domain": SAMPLES_PER_CLASS_PER_DOMAIN,
    "supcon_lr": SUPCON_LR,
    "supcon_temperature": SUPCON_TEMPERATURE,
    "same_domain_positive_weight": SAME_DOMAIN_POSITIVE_WEIGHT,
    "cross_domain_positive_weight": CROSS_DOMAIN_POSITIVE_WEIGHT,
    "val_n_way": VAL_N_WAY,
    "val_k_shot": VAL_K_SHOT,
    "val_q_query": VAL_Q_QUERY,
    "val_metric": VAL_METRIC,
    "val_temperature": VAL_TEMPERATURE,
    "early_stop_metric": EARLY_STOP_METRIC,
    "early_stop_patience_evals": EARLY_STOP_PATIENCE_EVALS,
    "early_stop_min_delta": EARLY_STOP_MIN_DELTA,
    "seed": SEED,
}


def latest_metrics(history):
    return {
        "step": history["step"][-1],
        "latest_contrastive_train_loss": history["contrastive_train_loss"][-1],
        "latest_contrastive_train_acc": history["contrastive_train_acc"][-1],
        "latest_source_val_loss": history["source_val_loss"][-1],
        "latest_source_val_acc": history["source_val_acc"][-1],
        "latest_target_val_loss": history["target_val_loss"][-1],
        "latest_target_val_acc": history["target_val_acc"][-1],
    }


def training_payload(history, *, stopped_early=False):
    return {
        "history": history,
        "config": supcon_config,
        "selection": {
            "best_metric": EARLY_STOP_METRIC,
            "best_metric_value": best_metric_value,
            "best_metric_step": best_metric_step,
            "stopped_early": stopped_early,
        },
    }


def save_latest_checkpoint(model, history, *, stopped_early=False):
    save_checkpoint(
        model,
        train_artifact_dir,
        labels=PERSONS,
        config=supcon_config,
        metrics={**latest_metrics(history), "stopped_early": stopped_early},
        history=history,
        name="latest.pt",
    )
    save_json(train_artifact_dir / "history.json", training_payload(history, stopped_early=stopped_early))


def save_best_checkpoint(model, history):
    if best_metric_step is None:
        raise ValueError("Cannot save best checkpoint before a post-training validation step.")
    save_checkpoint(
        model,
        run_dir(PROJECT_ROOT, MODEL_RUN_ID),
        labels=PERSONS,
        config=supcon_config,
        metrics={
            **latest_metrics(history),
            "best_metric": EARLY_STOP_METRIC,
            "best_metric_value": best_metric_value,
            "best_metric_step": best_metric_step,
        },
        history=history,
        name="model.pt",
    )
    refresh_run_checkpoint(PROJECT_ROOT, MODEL_RUN_ID)


def is_better_metric(value, best_value):
    return np.isfinite(value) and value > best_value + EARLY_STOP_MIN_DELTA


model = build_raw_encoder(device)
optimizer = torch.optim.Adam(model.parameters(), lr=SUPCON_LR)
batch_rng = np.random.default_rng(SEED)
source_val_rng = np.random.default_rng(SEED + 1)
target_val_rng = np.random.default_rng(SEED + 2)

history = {
    "step": [],
    "contrastive_train_loss": [],
    "contrastive_train_acc": [],
    "source_val_loss": [],
    "source_val_acc": [],
    "target_val_loss": [],
    "target_val_acc": [],
}

# Step-0 validation before any optimizer update.
source_val_metrics = eval_source_episodes(model, source_val_rng)
target_val_metrics = eval_target_episodes(model, target_val_rng)
history["step"].append(0)
history["contrastive_train_loss"].append(np.nan)
history["contrastive_train_acc"].append(np.nan)
history["source_val_loss"].append(source_val_metrics["loss"])
history["source_val_acc"].append(source_val_metrics["acc"])
history["target_val_loss"].append(target_val_metrics["loss"])
history["target_val_acc"].append(target_val_metrics["acc"])

best_metric_value = -np.inf
best_metric_step = None
evals_without_improvement = 0
stopped_early = False
save_latest_checkpoint(model, history)

print(f"step 0000/{TOTAL_STEPS}: source_acc={source_val_metrics['acc']:.4f} target_acc={target_val_metrics['acc']:.4f}")
show_live_curves(history)

progress = tqdm(range(1, TOTAL_STEPS + 1), desc="raw CSI SupCon training steps")
for step in progress:
    batch_indices = sample_contrastive_indices(
        raw_train_labels,
        raw_train_domains,
        classes_per_batch=CLASSES_PER_BATCH,
        samples_per_class_per_domain=SAMPLES_PER_CLASS_PER_DOMAIN,
        domains_per_class=DOMAINS_PER_CLASS,
        rng=batch_rng,
    )
    batch = load_contrastive_batch(raw_train_dataset, batch_indices, include_domains=True)
    train_metrics = contrastive_step(
        model,
        batch,
        optimizer,
        device=device,
        temperature=SUPCON_TEMPERATURE,
        same_domain_positive_weight=SAME_DOMAIN_POSITIVE_WEIGHT,
        cross_domain_positive_weight=CROSS_DOMAIN_POSITIVE_WEIGHT,
    )

    source_val_metrics = {"loss": np.nan, "acc": np.nan}
    target_val_metrics = {"loss": np.nan, "acc": np.nan}
    should_eval = step % EVAL_EVERY_STEPS == 0 or step == TOTAL_STEPS
    if should_eval:
        source_val_metrics = eval_source_episodes(model, source_val_rng)
        target_val_metrics = eval_target_episodes(model, target_val_rng)

    history["step"].append(step)
    history["contrastive_train_loss"].append(train_metrics["loss"])
    history["contrastive_train_acc"].append(train_metrics["batch_retrieval_acc"])
    history["source_val_loss"].append(source_val_metrics["loss"])
    history["source_val_acc"].append(source_val_metrics["acc"])
    history["target_val_loss"].append(target_val_metrics["loss"])
    history["target_val_acc"].append(target_val_metrics["acc"])

    progress.set_postfix(
        train_loss=f"{train_metrics['loss']:.4f}",
        train_acc=f"{train_metrics['batch_retrieval_acc']:.4f}",
        source_acc="nan" if np.isnan(source_val_metrics["acc"]) else f"{source_val_metrics['acc']:.4f}",
        target_acc="nan" if np.isnan(target_val_metrics["acc"]) else f"{target_val_metrics['acc']:.4f}",
    )

    if step % SAVE_EVERY_STEPS == 0 or step == TOTAL_STEPS:
        save_latest_checkpoint(model, history, stopped_early=stopped_early)

    if should_eval:
        current_metric_value = history[EARLY_STOP_METRIC][-1]
        if is_better_metric(current_metric_value, best_metric_value):
            best_metric_value = current_metric_value
            best_metric_step = step
            evals_without_improvement = 0
            save_best_checkpoint(model, history)
            print(f"New best {EARLY_STOP_METRIC}: {best_metric_value:.4f} at step {best_metric_step}")
        else:
            evals_without_improvement += 1
            print(f"No {EARLY_STOP_METRIC} improvement for {evals_without_improvement}/{EARLY_STOP_PATIENCE_EVALS} evals")

        if evals_without_improvement >= EARLY_STOP_PATIENCE_EVALS:
            stopped_early = True
            save_latest_checkpoint(model, history, stopped_early=stopped_early)
            print(f"Early stopping at step {step}. Best {EARLY_STOP_METRIC}={best_metric_value:.4f} at step {best_metric_step}.")
            break

    if step % LIVE_PLOT_EVERY_STEPS == 0 or step == TOTAL_STEPS:
        show_live_curves(history)
        print(f"step {step:04d}/{TOTAL_STEPS}: train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['batch_retrieval_acc']:.4f} source_acc={source_val_metrics['acc']:.4f} target_acc={target_val_metrics['acc']:.4f}")

save_latest_checkpoint(model, history, stopped_early=stopped_early)
best_checkpoint = torch.load(run_checkpoint_path(PROJECT_ROOT, MODEL_RUN_ID), map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint["model_state_dict"])
model.eval()

for dataset in (raw_train_dataset, raw_source_val_dataset, raw_target_enrollment_dataset, raw_target_query_dataset):
    dataset.clear_cache()
gc.collect()
if is_available():
    torch.cuda.empty_cache()

if best_metric_step is None:
    save_best_checkpoint(model, history)
    best_metric_step = history["step"][-1]
print(f"Loaded best model from step {best_metric_step}: {run_checkpoint_path(PROJECT_ROOT, MODEL_RUN_ID)}")

## Save Final Training Artifacts

In [ ]:
save_json(train_artifact_dir / "history.json", training_payload(history, stopped_early=stopped_early))
history_fig, _ = plot_step_curves(
    history,
    loss_keys=["contrastive_train_loss", "source_val_loss", "target_val_loss"],
    acc_keys=["contrastive_train_acc", "source_val_acc", "target_val_acc"],
    title="Raw CSI supervised contrastive training",
)
history_fig.savefig(train_artifact_dir / "curves.png", dpi=150)
plt.show()

print("Run directory:", run_dir(PROJECT_ROOT, MODEL_RUN_ID))
print("Best model:", run_checkpoint_path(PROJECT_ROOT, MODEL_RUN_ID))
print("Latest checkpoint:", train_artifact_dir / "latest.pt")
print("Training history:", train_artifact_dir / "history.json")
print("Training curves:", train_artifact_dir / "curves.png")

## Next Evaluation Command

In [3]:
import subprocess
import sys

script_path = PROJECT_ROOT / "scripts" / "evaluate_run.py"

PROJECTION_NAME = "umap_source_train_source_unseen_target_unseen"
COMPARISON_NAME = "umap_source_train_source_unseen_target_unseen_vs_baselines"

PROJECTION_PARTS = [
    "--projection-part", "source_train:PI-1a,PI-2a,PI-3a:0.0:0.6",
    "--projection-part", "source_unseen:PI-1a,PI-2a,PI-3a:0.8:1.0",
    "--projection-part", "target_unseen:PI-4a:0.8:1.0",
]

models = [
    {
        "model_run_id": "doppler_featuremap_proto",
        "model_key": "old_proto_featuremap",
    },
    {
        "model_run_id": "raw_csi_mixed_proto",
        "model_key": "raw_csi_proto",
    },
    {
        "model_run_id": "20260530_raw_csi_proto_domain_separated",
        "model_key": "raw_csi_proto",
    },
    {
        "model_run_id": MODEL_RUN_ID,
        "model_key": MODEL_KEY,
    },
]

# for model in models:
#     command = [
#         sys.executable,
#         str(script_path),
#         "--model-run-id", model["model_run_id"],
#         "--model-key", model["model_key"],
#         "--skip-kshot",
#         "--compute-umap",
#         "--projection-name", PROJECTION_NAME,
#         *PROJECTION_PARTS,
#     ]
#     subprocess.run(command, check=True, cwd=PROJECT_ROOT)

plot_command = [
    sys.executable,
    str(script_path),
    "--model-run-id", MODEL_RUN_ID,
    "--model-key", MODEL_KEY,
    "--skip-kshot",
    "--plot-umap",
    "--projection-name", PROJECTION_NAME,
    "--umap-comparison-name", COMPARISON_NAME,
    "--umap-baseline-run-ids",
    "raw_csi_mixed_proto",
    "20260530_raw_csi_proto_domain_separated",
    "doppler_featuremap_proto",
]

subprocess.run(plot_command, check=True, cwd=PROJECT_ROOT)

CompletedProcess(args=['c:\\Users\\gianm\\anaconda3\\envs\\wifi-doppler-har\\python.exe', 'C:\\Users\\gianm\\Development\\wifi-doppler-har\\scripts\\evaluate_run.py', '--model-run-id', '20260531_raw_csi_supcon_weighted_domain', '--model-key', 'raw_csi_proto', '--skip-kshot', '--plot-umap', '--projection-name', 'umap_source_train_source_unseen_target_unseen', '--umap-comparison-name', 'umap_source_train_source_unseen_target_unseen_vs_baselines', '--umap-baseline-run-ids', 'raw_csi_mixed_proto', '20260530_raw_csi_proto_domain_separated', 'doppler_featuremap_proto'], returncode=0)